# Lab 08 — 2 つの Data Agent を作成・公開

同じ業務定義（指示）で、データソースだけが異なる 2 つの Data Agent を作成します。

| Agent | 接続先 |
|---|---|
| `taxi_metrics_agent` | セマンティックモデル `sm_taxi` |
| `taxi_ontology_agent` | Ontology `onto_taxi`（コンテキストソース） |

設定内容はリポジトリの `config/agents.json` から `scripts/lab.py` が埋め込みます。何度実行しても同じ構成になります。

In [ ]:
%pip install -q "fabric-data-agent-sdk==0.1.36a0"

In [ ]:
import json
from datetime import datetime, timezone

def save_result(name, payload):
    """Persist a run summary that scripts/lab.py reads back from OneLake."""
    payload = {"notebook": name, "finishedAtUtc": datetime.now(timezone.utc).isoformat(timespec="seconds"), **payload}
    notebookutils.fs.put(f"Files/lab_results/{name}.json", json.dumps(payload, ensure_ascii=False, default=str), True)
    print(json.dumps(payload, ensure_ascii=False, indent=2, default=str))

checks = []

def check(name, expected, actual, tolerance=None):
    if tolerance is None:
        passed = expected == actual
    else:
        passed = actual is not None and abs(float(expected) - float(actual)) <= tolerance
    checks.append({"check": name, "expected": expected, "actual": actual, "passed": bool(passed)})

In [ ]:
import base64
import json
import warnings

from fabric.dataagent.client import create_data_agent

AGENTS = json.loads(base64.b64decode("{{AGENT_CONFIG_B64}}").decode("utf-8"))
warnings.filterwarnings("ignore", category=FutureWarning)

def select_all(datasource, root_id=None):
    """Select every element (tables, columns, measures) so the agent can query them."""
    token = None
    while True:
        page = datasource.get_elements(root_id=root_id, continuation_token=token)
        for element in page.get("value", []):
            if element.get("isSelected") is False:
                datasource.update_element(element["id"], is_selected=True)
            if element.get("hasChildren") or element.get("children"):
                select_all(datasource, element["id"])
        token = page.get("continuationToken")
        if not token:
            break

In [ ]:
summary = {}

metrics = AGENTS["metricsAgent"]
agent = create_data_agent(metrics["name"])
agent.update_settings(ai_instructions="\n".join(metrics["instructions"]))
existing = [ds for ds in agent.list_datasources()
            if ds.get_configuration().get("displayName") == metrics["source"]["item"]]
if not existing:
    agent.add_staging_datasource(metrics["source"]["item"], type=metrics["source"]["type"])
    existing = [ds for ds in agent.list_datasources()
                if ds.get_configuration().get("displayName") == metrics["source"]["item"]]
datasource = existing[0]
select_all(datasource)
datasource.update_configuration(description="NYC green taxi trips April 2026: fact_trips with vendor, payment, trip type, pickup and dropoff zone dimensions.")
agent.publish_staging(description=metrics["description"])
summary[metrics["name"]] = {"source": metrics["source"], "published": True}

In [ ]:
onto = AGENTS["ontologyAgent"]
agent = create_data_agent(onto["name"])
agent.update_settings(ai_instructions="\n".join(onto["instructions"]))
attached = [o for o in agent.list_ontologies() if o.get_configuration().get("displayName") == onto["source"]["item"]]
ontology = attached[0] if attached else agent.add_staging_ontology(onto["source"]["item"])
ontology.select_entities()
agent.publish_staging(description=onto["description"])
summary[onto["name"]] = {"source": onto["source"], "published": True}

save_result("nb_08_create_agents", {"passed": True, "agents": summary,
                                     "next": "scripts/lab.py agent-compare で 2 つの Agent に同じ質問をして比較します。"})